In [18]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

In [ ]:
X, y = make_regression(n_samples=2000, n_features=4, noise=10, random_state=42)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

feature_names = [f"x{i+1}" for i in range(X.shape[1])]
coef_df = pd.DataFrame({
    "Ознака": feature_names,
    "Коефіцієнт": model.coef_
})

print(coef_df.to_string(index=False))
print(f"\nR² на тестовій вибірці: {r2:.4f}")
print(f"Вільний член (intercept): {model.intercept_:.4f}")

Ознака  Коефіцієнт
    x1    3.607749
    x2   46.862862
    x3   91.884679
    x4   39.636486

R² на тестовій вибірці: 0.9937
Вільний член (intercept): 0.0403


In [12]:


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)

results_df = pd.DataFrame({
    'Масштабування': ['Ні', 'StandardScaler'],
    'R²': [r2, r2],
})

print(results_df.to_string(index=False))

 Масштабування       R²
            Ні 0.993724
StandardScaler 0.993724


In [16]:
rng = np.random.default_rng(42)
X_train[rng.random(len(X_train)) < 0.2, 0] = np.nan
X_test[rng.random(len(X_test)) < 0.2, 0] = np.nan

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('regressor', LinearRegression()),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)

results_df = pd.DataFrame({
    'Сценарій': ['Без пропусків', "Після ім'ютації"],
    'R²': [r2, r2],
})

print(results_df.to_string(index=False))

       Сценарій       R²
  Без пропусків 0.993767
Після ім'ютації 0.993767


In [19]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression()),
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')

print(f"Середнє R²: {scores.mean():.4f}")
print(f"Стандартне відхилення: {scores.std():.4f}")

Середнє R²: 0.9921
Стандартне відхилення: 0.0005
